In [2]:
import pandas as pd

def format_duration(duration):
    total_seconds = int(duration.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f'{hours:02}:{minutes:02}:{seconds:02}'

def check_tp_sl(price_data, signal_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None

    # Filter the subsequent prices correctly (inclusive of the signal_datetime)
    subsequent_prices = price_data.loc[signal_datetime:]
    print(f"Subsequent prices after {signal_datetime}:\n{subsequent_prices}")

    for current_datetime, price_row in subsequent_prices.iterrows():
        print(f"Checking price at {current_datetime}: High={price_row['High']}, Low={price_row['Low']}")
        
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                print(f"TP hit at {current_datetime}")
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                print(f"SL hit at {current_datetime}")
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                print(f"TP hit at {current_datetime}")
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                print(f"SL hit at {current_datetime}")
                break
    
    if exit_datetime:
        duration = exit_datetime - signal_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def backtest_trades(price_data, signal_data, tp=None, sl=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration'
    ])
    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        if signal_datetime not in price_data.index:
            print(f"No entry price found for signal datetime: {signal_datetime}")
            continue
        entry_price = price_data.at[signal_datetime, 'Open']
        print(f"Signal Datetime: {signal_datetime}, Side: {side}, Entry Price: {entry_price}")
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        print(f"TP Price: {tp_price}, SL Price: {sl_price}")
        
        result, duration_str = check_tp_sl(price_data, signal_datetime, tp_price, sl_price, side)

        print(f"Result: {result}, Duration: {duration_str}")
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

# Load the price data
price_data = pd.read_csv('E:\SignalModel\price 2024-05-01, 2024-06-01 min.csv')
price_data['Datetime'] = pd.to_datetime(price_data['Datetime'])
price_data.set_index('Datetime', inplace=True)
price_data.sort_index(ascending=True, inplace=True)

signal_data = pd.DataFrame({
    'Datetime': ['2024-05-20 21:00:00', '2024-05-21 17:00:00', '2024-05-21 21:00:00'],
    'Signal': [-1, -1, 1]
})

signal_data['Datetime'] = pd.to_datetime(signal_data['Datetime'])

results = backtest_trades(price_data, signal_data, tp=0.009, sl=0.016)
results


TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`